# Exploratory Data Analysis - Old Car Prices
Run from the `notebooks/` folder after `pip install -r ../backend/requirements.txt`.

In [ ]:
import sys
sys.path.insert(0, '../backend')
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt
from ml.preprocess import clean_dataset
raw = pd.read_csv('../data/car_price.csv')
raw.head()

## 1. Structure, types, missing values, duplicates

In [ ]:
print(raw.shape)
print(raw.dtypes)
print(raw.isna().sum())
print('duplicates (ignoring index col):', raw.drop(columns=['Unnamed: 0']).duplicated().sum())

## 2. Clean and parse (prices, km, engine, seats, brand/model)

In [ ]:
df, report = clean_dataset(raw)
print(report)
df.describe().T

## 3. Target distribution
Price is strongly right-skewed, so the model is trained on log(price).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df.price_inr/1e5, bins=60, ax=ax[0]); ax[0].set_title('Price (lakh)')
sns.histplot(np.log1p(df.price_inr), bins=60, ax=ax[1]); ax[1].set_title('log price')
plt.show()

## 4. Outliers (IQR rule)

In [ ]:
for c in ['price_inr','kms_driven','engine_cc']:
    q1,q3=df[c].quantile([.25,.75]); i=q3-q1
    print(c, int(((df[c]<q1-1.5*i)|(df[c]>q3+1.5*i)).sum()))
sns.boxplot(data=df[['kms_driven']]); plt.show()

## 5. Correlation

In [ ]:
num=['price_inr','kms_driven','engine_cc','manufacture_year','ownership','seats']
sns.heatmap(df[num].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0); plt.show()

## 6. Categorical drivers

In [ ]:
for c in ['fuel_type','transmission','ownership','seats']:
    print(df.groupby(c).price_inr.agg(['count','median']).round(0), '\n')
df.brand.value_counts().head(15)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.scatterplot(data=df, x='manufacture_year', y=np.log1p(df.price_inr), alpha=.3, ax=ax[0])
sns.scatterplot(data=df, x='kms_driven', y=np.log1p(df.price_inr), alpha=.3, ax=ax[1])
plt.show()